In [7]:
import pandas as pd
import numpy as np

# ==========================
# Load data
# ==========================
df = pd.read_csv("Downloads/Nifty 50 Historical Data.csv")

# Convert dates
df["Date"] = pd.to_datetime(
    df["Date"],
    dayfirst=True,
    format="mixed"
)

# Sort oldest → newest
df = df.sort_values("Date").reset_index(drop=True)

# Convert numerical columns
for col in ["Price", "High", "Low"]:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .astype(float)
    )

# ==========================
# Breakout levels
# ==========================
lookback = 20

df["HH20"] = df["High"].rolling(lookback).max().shift(1)
df["LL20"] = df["Low"].rolling(lookback).min().shift(1)

# ==========================
# Parameters
# ==========================
stop_loss = 0.03
take_profit = 0.06

holding = 0          # 0 = no stock, 1 = holding one stock
entry_price = None

signals = []

# ==========================
# Main Loop
# ==========================
for i in range(len(df)):

    # First 20 rows
    if pd.isna(df.loc[i, "HH20"]):
        signals.append(0)
        continue

    price = df.loc[i, "Price"]
    hh20 = df.loc[i, "HH20"]
    ll20 = df.loc[i, "LL20"]

    signal = 0

    # ======================================
    # If currently holding a stock
    # ======================================
    if holding == 1:

        ret = (price - entry_price) / entry_price

        # Sell conditions
        if (
            ret <= -stop_loss
            or ret >= take_profit
            or price < ll20
        ):
            signal = -1
            holding = 0
            entry_price = None

    # ======================================
    # If not holding any stock
    # ======================================
    else:

        # Buy condition
        if price > hh20:
            signal = 1
            holding = 1
            entry_price = price

    signals.append(signal)

# ==========================
# Output
# ==========================
result = pd.DataFrame({
    "Date": df["Date"].dt.strftime("%Y-%m-%d"),
    "Signal": signals
})

result.to_csv(
    "week.csv",
    index=False
)

print(result.head())
print(result.tail())

         Date  Signal
0  2016-01-01       0
1  2016-01-04       0
2  2016-01-05       0
3  2016-01-06       0
4  2016-01-07       0
            Date  Signal
2474  2025-12-26       0
2475  2025-12-29       0
2476  2025-12-30       0
2477  2025-12-31       0
2478  2026-01-01       0
